# Example 1 - Queues Only

In [19]:
import simpy
import vidigi

env = simpy.Environment()
logger = vidigi.logging.EventLogger(env=env)
treatment_cubicles = simpy.Resource(env, capacity=2)

def patient(env, patient_id, treatment_cubicles, logger):

    logger.log_arrival(entity_id=patient_id)

    logger.log_queue(entity_id=patient_id, event="treatment_wait_begins")

    with treatment_cubicles.request() as req:
        yield req
        logger.log_queue(entity_id=patient_id,
                         event="treatment_occurring")
        yield env.timeout(30)

    logger.log_departure(entity_id=patient_id)

def run_model(env, treatment_cubicles, logger):
    for i in range(50):
        env.process(
            patient(env, i,treatment_cubicles, logger)
           )
        yield env.timeout(5)

env.process(run_model(env, treatment_cubicles, logger))
env.run(until=500)

event_log = logger.to_dataframe()

event_log

,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id
0,0,arrival_departure,arrival,0.0,None,None,None,None
1,0,queue,treatment_wait_begins,0.0,None,None,None,None
2,0,queue,treatment_occurring,0.0,None,None,None,None
3,1,arrival_departure,arrival,5.0,None,None,None,None
4,1,queue,treatment_wait_begins,5.0,None,None,None,None
...,...,...,...,...,...,...,...,...
161,31,queue,treatment_occurring,455.0,None,None,None,None
162,30,arrival_departure,depart,480.0,None,None,None,None
163,32,queue,treatment_occurring,480.0,None,None,None,None
164,31,arrival_departure,depart,485.0,None,None,None,None


In [20]:
event_position_df = vidigi.utils.create_event_position_df([
    vidigi.utils.EventPosition(event='arrival', x=50, y=450, label="Arrival"),
    vidigi.utils.EventPosition(event='treatment_wait_begins', x=205, y=275, label="Waiting for Treatment"),
    vidigi.utils.EventPosition(event='treatment_begins', x=205, y=175, label="Being Treated", resource='n_cubicles'),
    vidigi.utils.EventPosition(event='depart', x=270, y=70, label="Exit")
])

event_position_df

,event,x,y,label,resource
0,arrival,50,450,Arrival,None
1,treatment_wait_begins,205,275,Waiting for Treatment,None
2,treatment_begins,205,175,Being Treated,n_cubicles
3,depart,270,70,Exit,None


In [21]:
class Params:
    def __init__(self):
        self.n_cubicles = 2

In [22]:
full_entity_df = vidigi.prep.reshape_for_animations(
        event_log=event_log,
        every_x_time_units=5,
        limit_duration=500,
        step_snapshot_max=125
)

full_entity_df


,index,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id,rank,snapshot_time
0,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,0
1,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,5
2,5,1,queue,treatment_occurring,5.0,None,None,None,None,2.0,5
3,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,10
4,5,1,queue,treatment_occurring,5.0,None,None,None,None,2.0,10
...,...,...,...,...,...,...,...,...,...,...,...
2300,121,45,queue,treatment_wait_begins,225.0,None,None,None,None,12.0,500
2301,123,46,queue,treatment_wait_begins,230.0,None,None,None,None,13.0,500
2302,125,47,queue,treatment_wait_begins,235.0,None,None,None,None,14.0,500
2303,128,48,queue,treatment_wait_begins,240.0,None,None,None,None,15.0,500


In [23]:
full_entity_df_plus_pos = vidigi.prep.generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_position_df,
    wrap_queues_at=25,
    gap_between_entities=6,
    gap_between_queue_rows=30,
    step_snapshot_max=125
)

full_entity_df_plus_pos

,index,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id,rank,snapshot_time,x,y_final,label,resource,x_final,row,y,icon
0,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,🧔🏼
1,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,5,NaN,NaN,NaN,NaN,NaN,0.0,NaN,🧔🏼
2,5,1,queue,treatment_occurring,5.0,None,None,None,None,2.0,5,NaN,NaN,NaN,NaN,NaN,0.0,NaN,👨🏿‍🦯
3,2,0,queue,treatment_occurring,0.0,None,None,None,None,1.0,10,NaN,NaN,NaN,NaN,NaN,0.0,NaN,🧔🏼
4,5,1,queue,treatment_occurring,5.0,None,None,None,None,2.0,10,NaN,NaN,NaN,NaN,NaN,0.0,NaN,👨🏿‍🦯
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2236,121,45,queue,treatment_wait_begins,225.0,None,None,None,None,12.0,500,205.0,275.0,Waiting for Treatment,None,139.0,0.0,NaN,👩🏽‍🎓
2237,123,46,queue,treatment_wait_begins,230.0,None,None,None,None,13.0,500,205.0,275.0,Waiting for Treatment,None,133.0,0.0,NaN,👱🏻‍♀️
2238,125,47,queue,treatment_wait_begins,235.0,None,None,None,None,14.0,500,205.0,275.0,Waiting for Treatment,None,127.0,0.0,NaN,👲🏼
2239,128,48,queue,treatment_wait_begins,240.0,None,None,None,None,15.0,500,205.0,275.0,Waiting for Treatment,None,121.0,0.0,NaN,🧕🏾


In [24]:
fig = vidigi.animation.generate_animation(
    full_entity_df_plus_pos=full_entity_df_plus_pos,
    event_position_df=event_position_df,
    scenario=Params(),
    entity_icon_size=20,
    resource_icon_size=20,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=300,
    override_y_max=500,
    start_time="07:00:00",
    time_display_units="simulation_day_clock_ampm",
    display_stage_labels=False,
    add_background_image=(
      "https://raw.githubusercontent.com/Bergam0t/" +
      "journal_of_simulation_25_paper_vidigi/refs/heads/main/bg.png"
    )
  )

fig

c:\journal_of_simulation_25_paper_vidigi\.venv\Lib\site-packages\vidigi\animation.py:239: FutureWarning:

The 'unit' keyword in TimedeltaIndex construction is deprecated and will be removed in a future version. Use pd.to_timedelta instead.



# Example 2 - With Resource Logging

In [25]:
import simpy
import vidigi

env = simpy.Environment()
logger = vidigi.logging.EventLogger(env=env)
treatment_cubicles = vidigi.resources.VidigiStore(env, num_resources=2)

def patient(env, patient_id, treatment_cubicles, logger):

    logger.log_arrival(entity_id=patient_id)

    logger.log_queue(entity_id=patient_id, event="treatment_wait_begins")

    with treatment_cubicles.request() as req:
        treatment_cubicle = yield req
        logger.log_resource_use_start(
            entity_id=patient_id, event="treatment_begins",
            resource_id=treatment_cubicle.id_attribute
            )

        yield env.timeout(30)

        logger.log_resource_use_end(
            entity_id=patient_id, event="treatment_complete",
            resource_id=treatment_cubicle.id_attribute
            )


    logger.log_departure(entity_id=patient_id)

def run_model(env, treatment_cubicles, logger):
    for i in range(50):
        env.process(
            patient(env, i,treatment_cubicles, logger)
           )
        yield env.timeout(5)

env.process(run_model(env, treatment_cubicles, logger))
env.run(until=500)

event_log = logger.to_dataframe()

event_log

,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id
0,0,arrival_departure,arrival,0.0,None,None,None,NaN
1,0,queue,treatment_wait_begins,0.0,None,None,None,NaN
2,0,resource_use,treatment_begins,0.0,None,None,None,1.0
3,1,arrival_departure,arrival,5.0,None,None,None,NaN
4,1,queue,treatment_wait_begins,5.0,None,None,None,NaN
...,...,...,...,...,...,...,...,...
193,30,arrival_departure,depart,480.0,None,None,None,NaN
194,32,resource_use,treatment_begins,480.0,None,None,None,1.0
195,31,resource_use_end,treatment_complete,485.0,None,None,None,2.0
196,31,arrival_departure,depart,485.0,None,None,None,NaN


In [26]:
event_position_df = vidigi.utils.create_event_position_df([
    vidigi.utils.EventPosition(event='arrival', x=50, y=450, label="Arrival"),
    vidigi.utils.EventPosition(event='treatment_wait_begins', x=205, y=275, label="Waiting for Treatment"),
    vidigi.utils.EventPosition(event='treatment_begins', x=205, y=175, label="Being Treated", resource='n_cubicles'),
    vidigi.utils.EventPosition(event='depart', x=270, y=70, label="Exit")
])

event_position_df

,event,x,y,label,resource
0,arrival,50,450,Arrival,None
1,treatment_wait_begins,205,275,Waiting for Treatment,None
2,treatment_begins,205,175,Being Treated,n_cubicles
3,depart,270,70,Exit,None


In [27]:
class Params:
    def __init__(self):
        self.n_cubicles = 2

In [28]:
full_entity_df = vidigi.prep.reshape_for_animations(
        event_log=event_log,
        every_x_time_units=5,
        limit_duration=500,
        step_snapshot_max=125
)

full_entity_df


,index,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id,rank,snapshot_time
0,2,0,resource_use,treatment_begins,0.0,None,None,None,1.0,1.0,0
1,2,0,resource_use,treatment_begins,0.0,None,None,None,1.0,1.0,5
2,5,1,resource_use,treatment_begins,5.0,None,None,None,2.0,2.0,5
3,2,0,resource_use,treatment_begins,0.0,None,None,None,1.0,1.0,10
4,5,1,resource_use,treatment_begins,5.0,None,None,None,2.0,2.0,10
...,...,...,...,...,...,...,...,...,...,...,...
2300,135,45,queue,treatment_wait_begins,225.0,None,None,None,NaN,12.0,500
2301,137,46,queue,treatment_wait_begins,230.0,None,None,None,NaN,13.0,500
2302,139,47,queue,treatment_wait_begins,235.0,None,None,None,NaN,14.0,500
2303,143,48,queue,treatment_wait_begins,240.0,None,None,None,NaN,15.0,500


In [29]:
full_entity_df_plus_pos = vidigi.prep.generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_position_df,
    wrap_queues_at=25,
    gap_between_entities=6,
    gap_between_queue_rows=30,
    step_snapshot_max=125
)

full_entity_df_plus_pos

,index,entity_id,event_type,event,time,pathway,run_number,timestamp,resource_id,rank,snapshot_time,x,y_final,label,resource,x_final,row,y,icon
0,7,2,queue,treatment_wait_begins,10.0,None,None,None,NaN,1.0,10,205,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
1,7,2,queue,treatment_wait_begins,10.0,None,None,None,NaN,1.0,15,205,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
2,9,3,queue,treatment_wait_begins,15.0,None,None,None,NaN,2.0,15,205,275.0,Waiting for Treatment,None,199.0,0.0,NaN,🧑🏻
3,7,2,queue,treatment_wait_begins,10.0,None,None,None,NaN,1.0,20,205,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
4,9,3,queue,treatment_wait_begins,15.0,None,None,None,NaN,2.0,20,205,275.0,Waiting for Treatment,None,199.0,0.0,NaN,🧑🏻
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2236,197,33,resource_use,treatment_begins,485.0,None,None,None,2.0,2.0,490,205,175.0,Being Treated,n_cubicles,195.0,0.0,NaN,👩🏿‍🎓
2237,194,32,resource_use,treatment_begins,480.0,None,None,None,1.0,1.0,495,205,175.0,Being Treated,n_cubicles,205.0,0.0,NaN,🙋🏽‍♂️
2238,197,33,resource_use,treatment_begins,485.0,None,None,None,2.0,2.0,495,205,175.0,Being Treated,n_cubicles,195.0,0.0,NaN,👩🏿‍🎓
2239,194,32,resource_use,treatment_begins,480.0,None,None,None,1.0,1.0,500,205,175.0,Being Treated,n_cubicles,205.0,0.0,NaN,🙋🏽‍♂️


In [30]:
fig = vidigi.animation.generate_animation(
    full_entity_df_plus_pos=full_entity_df_plus_pos,
    event_position_df=event_position_df,
    scenario=Params(),
    entity_icon_size=20,
    resource_icon_size=20,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=300,
    override_y_max=500,
    start_time="07:00:00",
    time_display_units="simulation_day_clock_ampm",
    display_stage_labels=False,
    add_background_image=(
      "https://raw.githubusercontent.com/Bergam0t/" +
      "journal_of_simulation_25_paper_vidigi/refs/heads/main/bg.png"
    )
  )

fig

c:\journal_of_simulation_25_paper_vidigi\.venv\Lib\site-packages\vidigi\animation.py:239: FutureWarning:

The 'unit' keyword in TimedeltaIndex construction is deprecated and will be removed in a future version. Use pd.to_timedelta instead.



# Example 3 - ciw

In [31]:
import ciw
import vidigi

# Create a simple model with 1 node containing 5 servers
N = ciw.create_network(
    arrival_distributions=[ciw.dists.Deterministic(value=5)],
    service_distributions=[ciw.dists.Deterministic(value=30)],
    number_of_servers=[2]
)

Q = ciw.Simulation(N)
Q.simulate_until_max_time(500)
event_log = vidigi.ciw.event_log_from_ciw_recs(
  Q.get_all_records(), node_name_list=["treatment"]
  )

event_log

,entity_id,pathway,event_type,event,time,resource_id
0,1,Model,arrival_departure,arrival,5,NaN
1,1,Model,queue,treatment_wait_begins,5,NaN
2,1,Model,resource_use,treatment_begins,5,1.0
3,1,Model,resource_use_end,treatment_ends,35,1.0
4,1,Model,arrival_departure,depart,35,NaN
...,...,...,...,...,...,...
155,32,Model,arrival_departure,arrival,160,NaN
156,32,Model,queue,treatment_wait_begins,160,NaN
157,32,Model,resource_use,treatment_begins,460,2.0
158,32,Model,resource_use_end,treatment_ends,490,2.0


In [32]:
event_position_df = vidigi.utils.create_event_position_df([
    vidigi.utils.EventPosition(event='arrival', x=50, y=450, label="Arrival"),
    vidigi.utils.EventPosition(event='treatment_wait_begins', x=205, y=275, label="Waiting for Treatment"),
    vidigi.utils.EventPosition(event='treatment_begins', x=205, y=175, label="Being Treated", resource='n_cubicles'),
    vidigi.utils.EventPosition(event='depart', x=270, y=70, label="Exit")
])

event_position_df

,event,x,y,label,resource
0,arrival,50,450,Arrival,None
1,treatment_wait_begins,205,275,Waiting for Treatment,None
2,treatment_begins,205,175,Being Treated,n_cubicles
3,depart,270,70,Exit,None


In [33]:
class Params:
    def __init__(self):
        self.n_cubicles = 2

In [34]:
full_entity_df = vidigi.prep.reshape_for_animations(
        event_log=event_log,
        every_x_time_units=5,
        limit_duration=500,
        step_snapshot_max=125
)

full_entity_df


,snapshot_time,index,entity_id,pathway,event_type,event,time,resource_id,rank
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,2.0,1.0,Model,resource_use,treatment_begins,5.0,1.0,1.0
2,10,2.0,1.0,Model,resource_use,treatment_begins,5.0,1.0,1.0
3,10,7.0,2.0,Model,resource_use,treatment_begins,10.0,2.0,2.0
4,15,2.0,1.0,Model,resource_use,treatment_begins,5.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...
1214,490,159.0,32.0,Model,arrival_departure,depart,490.0,NaN,1.0
1215,490,154.0,31.0,Model,arrival_departure,depart,485.0,NaN,1.0
1216,495,159.0,32.0,Model,arrival_departure,depart,490.0,NaN,1.0
1217,495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
full_entity_df_plus_pos = vidigi.prep.generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_position_df,
    wrap_queues_at=25,
    gap_between_entities=6,
    gap_between_queue_rows=30,
    step_snapshot_max=125
)

full_entity_df_plus_pos

,snapshot_time,index,entity_id,pathway,event_type,event,time,resource_id,rank,x,y_final,label,resource,x_final,row,y,icon
0,15,11.0,3.0,Model,queue,treatment_wait_begins,15.0,NaN,1.0,205.0,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
1,20,11.0,3.0,Model,queue,treatment_wait_begins,15.0,NaN,1.0,205.0,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
2,20,16.0,4.0,Model,queue,treatment_wait_begins,20.0,NaN,2.0,205.0,275.0,Waiting for Treatment,None,199.0,0.0,NaN,🧑🏻
3,25,11.0,3.0,Model,queue,treatment_wait_begins,15.0,NaN,1.0,205.0,275.0,Waiting for Treatment,None,205.0,0.0,NaN,👨🏻‍🦰
4,25,16.0,4.0,Model,queue,treatment_wait_begins,20.0,NaN,2.0,205.0,275.0,Waiting for Treatment,None,199.0,0.0,NaN,🧑🏻
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,480,157.0,32.0,Model,resource_use,treatment_begins,460.0,2.0,2.0,205.0,175.0,Being Treated,n_cubicles,195.0,0.0,NaN,👩🏼‍🦼
1151,485,157.0,32.0,Model,resource_use,treatment_begins,460.0,2.0,1.0,205.0,175.0,Being Treated,n_cubicles,195.0,0.0,NaN,👩🏼‍🦼
1152,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,🙋🏽‍♂️
1153,495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,🙋🏽‍♂️


In [36]:
fig = vidigi.animation.generate_animation(
    full_entity_df_plus_pos=full_entity_df_plus_pos,
    event_position_df=event_position_df,
    scenario=Params(),
    entity_icon_size=20,
    resource_icon_size=20,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=300,
    override_y_max=500,
    start_time="07:00:00",
    time_display_units="simulation_day_clock_ampm",
    display_stage_labels=False,
    add_background_image=(
      "https://raw.githubusercontent.com/Bergam0t/" +
      "journal_of_simulation_25_paper_vidigi/refs/heads/main/bg.png"
    )
  )

fig

c:\journal_of_simulation_25_paper_vidigi\.venv\Lib\site-packages\vidigi\animation.py:239: FutureWarning:

The 'unit' keyword in TimedeltaIndex construction is deprecated and will be removed in a future version. Use pd.to_timedelta instead.

